# Avendi V8-Y5 foundation and repair
Pinned commit: `7fe64684247c0c92762d7c03f04440e131e69757`


In [ ]:
SOURCE_SHA = '7fe64684247c0c92762d7c03f04440e131e69757'
OWNER_AUTHORIZED = True
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_DIR = '/tmp/avendi-y5-source'
CAMPAIGN_MODE = 'FULL'
SESSION_MINUTES = 480
RESUME_INPUT = None
DISCOVER_INPUTS = ['AVENDI_Y4_FULL.zip']
MIRROR_WORKING = False
EXPORT_GLOBS = ['*.zip', '*.sha256', 'FINAL_STATUS.json']
BACKEND_MODULES = ['experiments.avendi_y5.campaign']
INVENTORY_CMD = None
LAUNCH_CMD = ['python', '-m', 'experiments.avendi_y5.campaign', '--work-root', '/kaggle/working/y5-state', '--output', '/kaggle/working']
from pathlib import Path
import os, shutil, subprocess, sys
if REPO_URL:
    rp = Path(REPO_DIR)
    if not (rp / '.git').exists():
        subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(rp)])
    if subprocess.run(['git', 'fetch', '--depth', '1', 'origin', SOURCE_SHA], cwd=str(rp)).returncode != 0:
        subprocess.check_call(['git', 'fetch', 'origin'], cwd=str(rp))
    subprocess.check_call(['git', 'checkout', '--detach', SOURCE_SHA], cwd=str(rp))
    head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(rp), text=True).strip()
    assert head == SOURCE_SHA, f'HEAD mismatch: {head} != {SOURCE_SHA}'
    os.chdir(str(rp))
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))
if BACKEND_MODULES:
    _existing_bm = [x.strip() for x in os.environ.get('PLANTHIR_BACKEND_MODULES', '').split(',') if x.strip()]
    os.environ['PLANTHIR_BACKEND_MODULES'] = ','.join(dict.fromkeys([*_existing_bm, *BACKEND_MODULES]))
run_env = dict(os.environ)
run_env['PLANTHIR_CAMPAIGN_MODE'] = str(CAMPAIGN_MODE)
run_env['PLANTHIR_SESSION_MINUTES'] = str(SESSION_MINUTES)
if RESUME_INPUT is not None:
    run_env['PLANTHIR_RESUME_INPUT'] = str(RESUME_INPUT)
if not OWNER_AUTHORIZED:
    print('NOT LAUNCHED: set OWNER_AUTHORIZED = True before running GPU launch.')
    if INVENTORY_CMD:
        subprocess.check_call(INVENTORY_CMD, env=run_env)
else:
    if not REPO_URL:
        subprocess.check_call(['git', 'checkout', SOURCE_SHA])
    print('CHECKOUT_READY: proceeding to Preflight Gate & Authorized Launch cell.')


In [ ]:
REQUIRE_DUAL_GPU = True
PREFLIGHT_BACKEND = 'avendi_y5'
PREFLIGHT_CUDA_CANARY = False
BACKEND_MODULES = ['experiments.avendi_y5.campaign']
PREFLIGHT_COMMANDS = [['python', '-m', 'experiments.avendi_y5.campaign', '--preflight'], ['python', '-m', 'pytest', '-q', 'tests/test_avendi_y5_analysis.py', 'tests/test_avendi_y5_composition.py', 'tests/test_avendi_y5_information.py', 'tests/test_avendi_y5_reader.py', 'tests/test_avendi_y5_runtime.py']]
import os, shutil, subprocess, sys, torch
from pathlib import Path
if BACKEND_MODULES:
    _existing_bm = [x.strip() for x in os.environ.get('PLANTHIR_BACKEND_MODULES', '').split(',') if x.strip()]
    os.environ['PLANTHIR_BACKEND_MODULES'] = ','.join(dict.fromkeys([*_existing_bm, *BACKEND_MODULES]))
if REQUIRE_DUAL_GPU and globals().get('OWNER_AUTHORIZED', False):
    assert torch.cuda.device_count() >= 2, f'Expected >=2 GPUs, found {torch.cuda.device_count()}'
elif REQUIRE_DUAL_GPU:
    print(f'DRY-RUN (OWNER_AUTHORIZED=False): skipping dual-GPU assertion (visible GPUs={torch.cuda.device_count()})')
subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'doctor'])
for pf_cmd in PREFLIGHT_COMMANDS:
    subprocess.check_call(pf_cmd)
if PREFLIGHT_BACKEND:
    cdir = Path('work/preflight_canary')
    cdir.mkdir(parents=True, exist_ok=True)
    spec_p = cdir / 'spec.json'
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'init', '--backend', PREFLIGHT_BACKEND, '--out', str(spec_p)])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'validate', '--spec', str(spec_p), '--canary'])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'run', '--spec', str(spec_p), '--work-root', str(cdir / 'cpu'), '--canary', '--device', 'cpu'])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'verify-bundle', '--bundle-dir', str(cdir / 'cpu' / 'result_bundle')])
    subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'replay-bundle', '--bundle-dir', str(cdir / 'cpu' / 'result_bundle'), '--output', str(cdir / 'REPLAY_RECEIPT.json')])
    if PREFLIGHT_CUDA_CANARY and globals().get('OWNER_AUTHORIZED', False) and torch.cuda.is_available():
        subprocess.check_call([sys.executable, '-m', 'anra_v5.palantiri', 'run', '--spec', str(spec_p), '--work-root', str(cdir / 'cuda'), '--canary', '--device', 'cuda:0'])
if globals().get('OWNER_AUTHORIZED', False):
    env_map = dict(globals().get('run_env') or os.environ)
    if os.environ.get('PLANTHIR_BACKEND_MODULES'):
        env_map['PLANTHIR_BACKEND_MODULES'] = os.environ['PLANTHIR_BACKEND_MODULES']
    env_map['PLANTHIR_CAMPAIGN_MODE'] = str(globals().get('CAMPAIGN_MODE', 'FULL'))
    env_map['PLANTHIR_SESSION_MINUTES'] = str(globals().get('SESSION_MINUTES', 300))
    if globals().get('RESUME_INPUT') is not None:
        env_map['PLANTHIR_RESUME_INPUT'] = str(globals()['RESUME_INPUT'])
    disc_inputs = globals().get('DISCOVER_INPUTS', ['AVENDI_Y4_FULL.zip'])
    launch_cmd = globals().get('LAUNCH_CMD', ['python', '-m', 'experiments.avendi_y5.campaign', '--work-root', '/kaggle/working/y5-state', '--output', '/kaggle/working'])
    mirror_work = globals().get('MIRROR_WORKING', False)
    exp_globs = globals().get('EXPORT_GLOBS', ['*.zip', '*.sha256', 'FINAL_STATUS.json'])
    if disc_inputs and Path('/kaggle/input').is_dir():
        for name in disc_inputs:
            found = sorted(Path('/kaggle/input').rglob(name))
            print('KAGGLE_INPUT', name, [str(p) for p in found])
    try:
        subprocess.check_call(launch_cmd, env=env_map)
    except BaseException as _exc:
        import json as _json
        Path('CRASH_RECEIPT.json').write_text(_json.dumps({'schema': 'anra-planthir-crash-receipt-v1', 'error_type': type(_exc).__name__, 'error': str(_exc), 'launch_cmd': list(launch_cmd)}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
        raise
    finally:
        if mirror_work and Path('/kaggle/working').is_dir():
            _mirror_pats = list(dict.fromkeys([*exp_globs, 'CRASH_RECEIPT.json', 'REGISTERED_PLAN_STATUS.json', 'CAMPAIGN_STATUS.json', 'JOB_RECEIPTS.json', 'PENDING_QUEUE.json']))
            for pat in _mirror_pats:
                for p in sorted(Path('.').glob(pat)):
                    if p.is_file():
                        dst = Path('/kaggle/working') / p.name
                        if p.resolve() != dst.resolve():
                            if dst.exists():
                                dst.unlink()
                            try:
                                os.link(p, dst)
                            except OSError:
                                shutil.copy2(p, dst)


In [ ]:
LINK_GLOBS = ['*.zip', '*.sha256', 'FINAL_STATUS.json']
from pathlib import Path
try:
    from IPython.display import FileLink, display
except Exception:
    FileLink = None
wdir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
for pat in LINK_GLOBS:
    for f in sorted(wdir.glob(pat)):
        if FileLink is not None:
            display(FileLink(str(f)))
        else:
            print('ARTIFACT:', f)
